In [453]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [454]:
df = pd.read_csv("Raisin_Dataset.csv")
df

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
0,87524,442.246011,253.291155,0.819738,90546,0.758651,1184.040,Kecimen
1,75166,406.690687,243.032436,0.801805,78789,0.684130,1121.786,Kecimen
2,90856,442.267048,266.328318,0.798354,93717,0.637613,1208.575,Kecimen
3,45928,286.540559,208.760042,0.684989,47336,0.699599,844.162,Kecimen
4,79408,352.190770,290.827533,0.564011,81463,0.792772,1073.251,Kecimen
...,...,...,...,...,...,...,...,...
895,83248,430.077308,247.838695,0.817263,85839,0.668793,1129.072,Besni
896,87350,440.735698,259.293149,0.808629,90899,0.636476,1214.252,Besni
897,99657,431.706981,298.837323,0.721684,106264,0.741099,1292.828,Besni
898,93523,476.344094,254.176054,0.845739,97653,0.658798,1258.548,Besni


In [455]:
print(df.isnull().sum())

Area               0
MajorAxisLength    0
MinorAxisLength    0
Eccentricity       0
ConvexArea         0
Extent             0
Perimeter          0
Class              0
dtype: int64


In [456]:
df['Class'] = df['Class'].map({'Kecimen': 0, 'Besni':1})

In [457]:
df

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
0,87524,442.246011,253.291155,0.819738,90546,0.758651,1184.040,0
1,75166,406.690687,243.032436,0.801805,78789,0.684130,1121.786,0
2,90856,442.267048,266.328318,0.798354,93717,0.637613,1208.575,0
3,45928,286.540559,208.760042,0.684989,47336,0.699599,844.162,0
4,79408,352.190770,290.827533,0.564011,81463,0.792772,1073.251,0
...,...,...,...,...,...,...,...,...
895,83248,430.077308,247.838695,0.817263,85839,0.668793,1129.072,1
896,87350,440.735698,259.293149,0.808629,90899,0.636476,1214.252,1
897,99657,431.706981,298.837323,0.721684,106264,0.741099,1292.828,1
898,93523,476.344094,254.176054,0.845739,97653,0.658798,1258.548,1


In [458]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [459]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import time

In [460]:
scaler = MinMaxScaler()
scaled_X = scaler.fit_transform(X)

In [461]:
chi_scores, p_value = chi2(scaled_X, y)

In [462]:
res = pd.DataFrame({"Features": X.columns, "Chi-2 Score": chi_scores, "P-Value": p_value})
res = res.sort_values(by="Chi-2 Score",ascending=False)
print(res)

          Features  Chi-2 Score       P-Value
0             Area    40.913845  1.590901e-10
4       ConvexArea    35.662379  2.346520e-09
1  MajorAxisLength    34.625936  3.995473e-09
6        Perimeter    26.290246  2.937686e-07
2  MinorAxisLength    14.726018  1.243189e-04
3     Eccentricity     5.311431  2.118595e-02
5           Extent     0.422287  5.157982e-01


In [463]:
least_imp = res.iloc[-1]["Features"]
print("Least important feature:", least_imp)

Least important feature: Extent


In [464]:
X = X.drop(columns=[least_imp])
print(X.columns)

Index(['Area', 'MajorAxisLength', 'MinorAxisLength', 'Eccentricity',
       'ConvexArea', 'Perimeter'],
      dtype='str')


In [465]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1, stratify = y)

In [466]:
print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

Training Features: (720, 6)
Testing Features: (180, 6)


In [467]:
def evalute(model, X_test, y_test):
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nAccuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)

    return accuracy, precision, recall, f1

#Decision


In [468]:
dt = DecisionTreeClassifier(random_state=1)
start_time = time.time()
dt.fit(X_train, y_train)
dt_time = time.time() - start_time
print("Decision tree results")
dt_accuracy, dt_precision, dt_recall, dt_f1 = evalute(
    dt,
    X_test,
    y_test
)

Decision tree results
Confusion Matrix:
[[73 17]
 [21 69]]

Accuracy: 0.7888888888888889
Precision: 0.8023255813953488
Recall: 0.7666666666666667
F1 Score: 0.7840909090909091


In [469]:
print("Training time:", dt_time, "seconds")

Training time: 0.004820585250854492 seconds


#adaBoost


In [470]:
ada = AdaBoostClassifier(random_state=1)
start_time = time.time()
ada.fit(X_train, y_train)
ada_time = time.time() - start_time
print("Ada result")
ada_accuracy, ada_precision, ada_recall, ada_f1 = evalute(ada, X_test, y_test)
print("Training time: ", ada_time, "seconds")

Ada result
Confusion Matrix:
[[82  8]
 [23 67]]

Accuracy: 0.8277777777777777
Precision: 0.8933333333333333
Recall: 0.7444444444444445
F1 Score: 0.8121212121212121
Training time:  0.09793639183044434 seconds


In [471]:
stump = DecisionTreeClassifier(max_depth = 1, random_state = 1)
deep_tree = DecisionTreeClassifier(max_depth = 3, random_state= 1)

In [472]:
ada_model = AdaBoostClassifier(random_state=1)
param_grid_data = {"n_estimators": [50,100,200],"learning_rate": [0.01, 0.1, 1],"estimator": [stump, deep_tree]
                  }

In [473]:
grid_ada = GridSearchCV(estimator = ada_model,
                        param_grid = param_grid_data,
                        cv = 5,
                        scoring = "f1",
                        n_jobs = 16
                       )
start_time = time.time()
grid_ada.fit(X_train, y_train)
ada_tuning_time = time.time() - start_time
print("Best Parameters" , grid_ada.best_params_)
print("\nF1 Score:" , grid_ada.best_score_)

Best Parameters {'estimator': DecisionTreeClassifier(max_depth=1, random_state=1), 'learning_rate': 1, 'n_estimators': 50}

F1 Score: 0.8653912203816161


In [474]:
best_ada = grid_ada.best_estimator_
print("Best AdaBoost Model Results")
best_ada_accuracy, best_ada_precision, best_ada_recall, best_ada_f1 = evalute(
    best_ada,
    X_test,
    y_test
)
print("Training and tuning time:", ada_tuning_time)

Best AdaBoost Model Results
Confusion Matrix:
[[82  8]
 [23 67]]

Accuracy: 0.8277777777777777
Precision: 0.8933333333333333
Recall: 0.7444444444444445
F1 Score: 0.8121212121212121
Training and tuning time: 2.253181219100952


#gradientBoost

In [475]:
gb = GradientBoostingClassifier(random_state = 1)
start_time = time.time()
gb.fit(X_train, y_train)
gb_time = time.time() - start_time
print("Gradient Boosting results")

gb_accuracy, gb_precision, gb_recall, gb_f1 = evalute(
    gb,
    X_test,
    y_test
)
print("Training time", gb_time)

Gradient Boosting results
Confusion Matrix:
[[81  9]
 [20 70]]

Accuracy: 0.8388888888888889
Precision: 0.8860759493670886
Recall: 0.7777777777777778
F1 Score: 0.8284023668639053
Training time 0.12269806861877441


In [ ]:
gb_model = GradientBoostingClassifier(random_state=1)
param_grid_gb = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.01, 0.1, 0.5, 1],
    "max_depth": [3, 4, 5],
    "subsample": [0.8, 1.0]
}

In [477]:
grid_gb = GridSearchCV(
                        estimator=gb_model,
                        param_grid=param_grid_gb,
                        cv=5,
                        scoring="f1",
                        n_jobs=-1
                        )

In [478]:
start_time = time.time()
grid_gb.fit(X_train, y_train)
gb_tuning_time = time.time() - start_time
print("Best Gradient Boosting Parameters:")
print(grid_gb.best_params_)
print("\nF1 Score:")
print(grid_gb.best_score_)

Best Gradient Boosting Parameters:
{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 50, 'subsample': 0.8}

F1 Score:
0.8612339168176742


In [479]:
best_gb = grid_gb.best_estimator_
print("Best Gradient Boosting Results")
best_gb_accuracy, best_gb_precision, best_gb_recall, best_gb_f1 = evalute(
    best_gb,
    X_test,
    y_test
)
print("Training/Tuning Time:", gb_tuning_time, "seconds")

Best Gradient Boosting Results
Confusion Matrix:
[[80 10]
 [18 72]]

Accuracy: 0.8444444444444444
Precision: 0.8780487804878049
Recall: 0.8
F1 Score: 0.8372093023255814
Training/Tuning Time: 6.792872905731201 seconds


In [480]:
comparison = pd.DataFrame({
    "Model": ["Decision Tree","AdaBoost","Gradient Boosting"],
    "Accuracy": [dt_accuracy,best_ada_accuracy,best_gb_accuracy],
    "Precision": [dt_precision,best_ada_precision,best_gb_precision],
    "Recall": [dt_recall,best_ada_recall,best_gb_recall],
    "F1 Score": [dt_f1,best_ada_f1,best_gb_f1],
    "Training Time (sec)": [dt_time,ada_tuning_time,gb_tuning_time]
})
print(comparison)

               Model  Accuracy  Precision    Recall  F1 Score  \
0      Decision Tree  0.788889   0.802326  0.766667  0.784091   
1           AdaBoost  0.827778   0.893333  0.744444  0.812121   
2  Gradient Boosting  0.844444   0.878049  0.800000  0.837209   

   Training Time (sec)  
0             0.004821  
1             2.253181  
2             6.792873  
